<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_13_2_text2speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 13: Speech Processing**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://github.com/jeffheaton/app_generative_ai).

# Module 13 Material

Module 13: Speech Processing

* Part 13.1: Intro to Speech Processing [[Video]](https://www.youtube.com/watch?v=ILNcv9zrMyQ&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_13_1_speech_models.ipynb)
* **Part 13.2: Text to Speech** [[Video]](https://www.youtube.com/watch?v=O5_oaK5fHqI&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_13_2_text2speech.ipynb)
* Part 13.3: Speech to Text [[Video]](https://www.youtube.com/watch?v=zor64w90fpQ&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_13_3_speech2text.ipynb)
* Part 13.4: Speech Bot [[Video]](https://www.youtube.com/watch?v=8fgxX6yLorI&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_13_4_speechbot.ipynb)
* Part 13.5: Future Directions in GenAI [[Video]](https://www.youtube.com/watch?v=T4AYP_XXTbg&ab_channel=JeffHeaton) [[Notebook]](t81_559_class_13_5_future.ipynb)


# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [1]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai openai pydub

Note: using Google CoLab
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 14.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


# Part 13.2: Text to Speech

In this module, we'll explore the fascinating world of text-to-speech (TTS) Large Language Models (LLMs), focusing on OpenAI's cutting-edge offerings. We'll primarily utilize OpenAI's gpt-4o-mini-tts model, a powerful and versatile tool designed for converting written text into natural-sounding speech. The model is optimized for real-time applications, making it ideal for scenarios that require low-latency audio generation. This model represents a significant advancement in speech synthesis technology, leveraging deep learning techniques to produce high-quality, lifelike vocal outputs. By delving into gpt-4o-mini-tts, we'll explore its capabilities, examine its practical applications, and understand how it's revolutionizing various industries, from accessibility solutions to interactive voice responses and beyond.


### Simple Text to Speech Example

his code snippet demonstrates how to use OpenAI's text-to-speech API to generate spoken audio from text. First, it imports the necessary libraries: openai for API interaction, IPython.display for audio playback in Jupyter notebooks, and base64 for encoding. The TEXT variable contains the message to be converted to speech. The openai.audio.speech.create() function is called with four parameters: the model ("gpt-4o-mini-tts"), the voice ("alloy"), the input text, and an optional instructions string that steers the delivery. OpenAI offers thirteen voice options, including:

* **alloy** - neutral
* **echo** - young
* **fable** - male
* **onyx** - deep male
* **nova** - female
* **shimmer** - warm female

Each voice has its unique characteristics, allowing users to choose the most suitable one for their application. The earlier model generations (tts-1 and tts-1-hd) remain available for now but support fewer voices and no instructions parameter; new work should use gpt-4o-mini-tts. The function returns a response object, from which the audio content is extracted and stored in the audio_data variable for further processing or playback.

In [2]:
import openai
import IPython.display as ipd
import base64

TEXT = "Hello there, I am one of the OpenAI chat voices, how are you?"

response = openai.audio.speech.create(
  model="gpt-4o-mini-tts",
  voice="alloy",
  input=TEXT,
  instructions="Speak in a cheerful, friendly tone."
)

# Get the audio content
audio_data = response.content

Unlike the older tts-1 model, gpt-4o-mini-tts also accepts an optional instructions parameter, which lets you direct the voice's tone, accent, pacing, or emotion in plain English, as the code above demonstrates. We can play this audio to the CoLab notebook user.

In [3]:
from IPython.display import Audio, display

# Play the audio in Colab
print("Playing audio:")
display(Audio(audio_data, autoplay=True))

Playing audio:


We can also save an audio file.

In [4]:
with open("audio.mp3", "wb") as f:
    f.write(audio_data)


We can download this audio file.

In [5]:
# prompt: How do I download an audio file I generated named audio.mp3?

from google.colab import files
files.download('audio.mp3')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Multiple Voices and Samples

The code demonstrates how to concatenate multiple text-to-speech responses from OpenAI's API, showcasing each of the available voices. It uses the pydub library to combine audio segments seamlessly. The script iterates through the thirteen voices available for gpt-4o-mini-tts, generating a sample audio for each voice saying "Hello, I am the [voice] voice." These individual audio segments are then combined into a single audio file using AudioSegment from pydub. The resulting audio plays each voice sample in sequence, allowing listeners to hear the distinct characteristics of each voice option. This approach is particularly useful for comparing different voices or creating a demo reel of available voice options in a single, continuous audio stream

In [6]:
import io
from openai import OpenAI
from IPython.display import Audio, display
from google.colab import files
import os

# Initialize OpenAI client
client = OpenAI()

# The thirteen voices supported by gpt-4o-mini-tts
voices = ["alloy", "ash", "ballad", "coral", "echo", "fable", "nova",
          "onyx", "sage", "shimmer", "verse", "marin", "cedar"]
audio_segments = []

for voice in voices:
    text = f"Hello, I am the {voice} voice."
    response = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice=voice,
        input=text
    )
    audio_segments.append(response.content)

# Combine audio segments
from pydub import AudioSegment

combined_audio = AudioSegment.empty()
for segment in audio_segments:
    audio = AudioSegment.from_mp3(io.BytesIO(segment))
    combined_audio += audio

# Convert the combined audio to a byte stream
buffer = io.BytesIO()
combined_audio.export(buffer, format="mp3")
buffer.seek(0)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


0

Play the audio to the CoLab user.

In [7]:
# Play the audio in Colab
print("Playing audio:")
display(Audio(buffer.read(), autoplay=True))

Playing audio:


Save the audio to a file.

In [8]:
# Reset buffer position
buffer.seek(0)

# Save the audio file
output_filename = "combined_voices.mp3"
with open(output_filename, "wb") as f:
    f.write(buffer.getvalue())

print(f"\nAudio saved as {output_filename}")


Audio saved as combined_voices.mp3


Download the audio file

In [9]:
files.download(output_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>